In [1]:
import numpy as np
import pandas as pd 

from sklearn.datasets import fetch_openml

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import LabelEncoder , OneHotEncoder ,OrdinalEncoder, MinMaxScaler, StandardScaler
import category_encoders as ce

from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split , GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB

from sklearn.metrics import confusion_matrix , ConfusionMatrixDisplay, accuracy_score , recall_score , precision_score , f1_score , roc_auc_score , classification_report

In [2]:
#telco-customer-churn 
X , y = fetch_openml(
    name='telco-customer-churn',
    as_frame = True, 
    return_X_y = True,

)

/Users/ola/Mystro/V2_ML_night/Telco_customer_churn/.venv/lib/python3.11/site-packages/sklearn/datasets/_openml.py:350: UserWarning: Multiple active versions of the dataset matching the name telco-customer-churn exist. Versions may be fundamentally different, returning version 1. Available versions:
- version 1, status: active
  url: https://www.openml.org/search?type=data&id=42178
- version 2, status: active
  url: https://www.openml.org/search?type=data&id=45568

  warn(warning_msg)


In [3]:
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,'No phone service',DSL,No,Yes,No,No,No,No,Month-to-month,Yes,'Electronic check',29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,'One year',No,'Mailed check',56.95,1889.5
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,'Mailed check',53.85,108.15
3,Male,0,No,No,45,No,'No phone service',DSL,Yes,No,Yes,Yes,No,No,'One year',No,'Bank transfer (automatic)',42.30,1840.75
4,Female,0,No,No,2,Yes,No,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Electronic check',70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,'One year',Yes,'Mailed check',84.80,1990.5
7039,Female,0,Yes,Yes,72,Yes,Yes,'Fiber optic',No,Yes,Yes,No,Yes,Yes,'One year',Yes,'Credit card (automatic)',103.20,7362.9
7040,Female,0,Yes,Yes,11,No,'No phone service',DSL,Yes,No,No,No,No,No,Month-to-month,Yes,'Electronic check',29.60,346.45
7041,Male,1,Yes,No,4,Yes,Yes,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Mailed check',74.40,306.6


In [4]:
y

0        No
1        No
2       Yes
3        No
4       Yes
       ... 
7038     No
7039     No
7040     No
7041    Yes
7042     No
Name: Churn, Length: 7043, dtype: object

In [5]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [6]:
X['SeniorCitizen'] = X['SeniorCitizen'].astype('category')

In [7]:
X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors = "coerce")

In [8]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7043 non-null   object  
 1   SeniorCitizen     7043 non-null   category
 2   Partner           7043 non-null   object  
 3   Dependents        7043 non-null   object  
 4   tenure            7043 non-null   int64   
 5   PhoneService      7043 non-null   object  
 6   MultipleLines     7043 non-null   object  
 7   InternetService   7043 non-null   object  
 8   OnlineSecurity    7043 non-null   object  
 9   OnlineBackup      7043 non-null   object  
 10  DeviceProtection  7043 non-null   object  
 11  TechSupport       7043 non-null   object  
 12  StreamingTV       7043 non-null   object  
 13  StreamingMovies   7043 non-null   object  
 14  Contract          7043 non-null   object  
 15  PaperlessBilling  7043 non-null   object  
 16  PaymentMethod     7043 n

In [9]:
cat = X.select_dtypes('object').columns
cat

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [10]:
for i in cat :
    X[i] = X[i].astype('category')
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7043 non-null   category
 1   SeniorCitizen     7043 non-null   category
 2   Partner           7043 non-null   category
 3   Dependents        7043 non-null   category
 4   tenure            7043 non-null   int64   
 5   PhoneService      7043 non-null   category
 6   MultipleLines     7043 non-null   category
 7   InternetService   7043 non-null   category
 8   OnlineSecurity    7043 non-null   category
 9   OnlineBackup      7043 non-null   category
 10  DeviceProtection  7043 non-null   category
 11  TechSupport       7043 non-null   category
 12  StreamingTV       7043 non-null   category
 13  StreamingMovies   7043 non-null   category
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   category
 16  PaymentMethod     7043 n

In [11]:
le = LabelEncoder()
y = le.fit_transform(y)
le_target = le

In [12]:
y

array([0, 0, 1, ..., 0, 1, 0], shape=(7043,))

In [13]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7043 non-null   category
 1   SeniorCitizen     7043 non-null   category
 2   Partner           7043 non-null   category
 3   Dependents        7043 non-null   category
 4   tenure            7043 non-null   int64   
 5   PhoneService      7043 non-null   category
 6   MultipleLines     7043 non-null   category
 7   InternetService   7043 non-null   category
 8   OnlineSecurity    7043 non-null   category
 9   OnlineBackup      7043 non-null   category
 10  DeviceProtection  7043 non-null   category
 11  TechSupport       7043 non-null   category
 12  StreamingTV       7043 non-null   category
 13  StreamingMovies   7043 non-null   category
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   category
 16  PaymentMethod     7043 n

In [14]:
num = X.select_dtypes(include=['int64', 'float64']).columns
cat = X.select_dtypes(include=['object' , 'category']).columns
print(f"Numerical Col : {num} \n____________ \nCategorical columns {cat}")

Numerical Col : Index(['tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object') 
____________ 
Categorical columns Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


In [15]:
num_pipe = Pipeline( # tuple that includes a list of tuples { dictionary }
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'mean')) ,
        ( 'scaler' , StandardScaler() )   
]) 

cat_pipe = Pipeline( 
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'most_frequent')) , # mode = most_frequent
        ( 'encoder' , OneHotEncoder(handle_unknown='ignore') )   
]) 

preprocessor = ColumnTransformer(
    transformers = [
        ('num' , num_pipe , num ), 
        ('cat' , cat_pipe , cat )
])

model = Pipeline(
    steps = [
        ("preprocessor" , preprocessor) , 
        ('classifier' , BernoulliNB() )
    ]
)

In [16]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.25 , random_state = 42)

In [17]:
model.fit(X_train , y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

In [18]:
y_pred = model.predict(X_test)

In [19]:
print(classification_report(y_test , y_pred ))

              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1282
           1       0.50      0.82      0.62       479

    accuracy                           0.73      1761
   macro avg       0.71      0.76      0.71      1761
weighted avg       0.80      0.73      0.74      1761



# **` ( TASK FOR BUSINESS CLASS ) `**
## 1- Recap EDA Principles on this data 
## 2-  Smote to handle imbalance 


In [20]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [21]:
num_pipe = Pipeline( 
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'mean')) ,
        ( 'scaler' , StandardScaler() )   
]) 

cat_pipe = Pipeline( 
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'most_frequent')) ,
        ( 'encoder' , OneHotEncoder(handle_unknown='ignore') )   
]) 

preprocessor = ColumnTransformer(
    transformers = [
        ('num' , num_pipe , num ), 
        ('cat' , cat_pipe , cat )
])

model_smote = Pipeline(
    steps = [
        ("preprocessor" , preprocessor) , 
        ('smote' , SMOTE(random_state = 42) ) ,
        ('classifier' , BernoulliNB() )
    ]
)

In [22]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7043 non-null   category
 1   SeniorCitizen     7043 non-null   category
 2   Partner           7043 non-null   category
 3   Dependents        7043 non-null   category
 4   tenure            7043 non-null   int64   
 5   PhoneService      7043 non-null   category
 6   MultipleLines     7043 non-null   category
 7   InternetService   7043 non-null   category
 8   OnlineSecurity    7043 non-null   category
 9   OnlineBackup      7043 non-null   category
 10  DeviceProtection  7043 non-null   category
 11  TechSupport       7043 non-null   category
 12  StreamingTV       7043 non-null   category
 13  StreamingMovies   7043 non-null   category
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   category
 16  PaymentMethod     7043 n

In [23]:
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,'No phone service',DSL,No,Yes,No,No,No,No,Month-to-month,Yes,'Electronic check',29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,'One year',No,'Mailed check',56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,'Mailed check',53.85,108.15
3,Male,0,No,No,45,No,'No phone service',DSL,Yes,No,Yes,Yes,No,No,'One year',No,'Bank transfer (automatic)',42.30,1840.75
4,Female,0,No,No,2,Yes,No,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Electronic check',70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,'One year',Yes,'Mailed check',84.80,1990.50
7039,Female,0,Yes,Yes,72,Yes,Yes,'Fiber optic',No,Yes,Yes,No,Yes,Yes,'One year',Yes,'Credit card (automatic)',103.20,7362.90
7040,Female,0,Yes,Yes,11,No,'No phone service',DSL,Yes,No,No,No,No,No,Month-to-month,Yes,'Electronic check',29.60,346.45
7041,Male,1,Yes,No,4,Yes,Yes,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Mailed check',74.40,306.60


In [24]:
X.loc[:, 'TotalCharges'] = X['TotalCharges'].fillna(X['TotalCharges'].mean()) #Abdullah

In [25]:
cat

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [26]:
# Keep the original category values so the label encoding can be reproduced (e.g. in Streamlit)
label_mappings = {c: sorted(X[c].unique().tolist()) for c in cat}

In [27]:
le = LabelEncoder()
for i in cat :
    X[i] = le.fit_transform(X[i])
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,1,0,0,1,1,2,1,1,1,1,2,1,2,29.85,29.85
1,1,0,0,0,34,1,1,1,2,1,2,1,1,1,0,0,3,56.95,1889.50
2,1,0,0,0,2,1,1,1,2,2,1,1,1,1,2,1,3,53.85,108.15
3,1,0,0,0,45,0,0,1,2,1,2,2,1,1,0,0,0,42.30,1840.75
4,0,0,0,0,2,1,1,0,1,1,1,1,1,1,2,1,2,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,2,1,2,1,2,2,2,2,0,1,3,84.80,1990.50
7039,0,0,1,1,72,1,2,0,1,2,2,1,2,2,0,1,1,103.20,7362.90
7040,0,0,1,1,11,0,0,1,2,1,1,1,1,1,2,1,2,29.60,346.45
7041,1,1,1,0,4,1,2,0,1,1,1,1,1,1,2,1,3,74.40,306.60


In [28]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   int64  
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   int64  
 3   Dependents        7043 non-null   int64  
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   int64  
 6   MultipleLines     7043 non-null   int64  
 7   InternetService   7043 non-null   int64  
 8   OnlineSecurity    7043 non-null   int64  
 9   OnlineBackup      7043 non-null   int64  
 10  DeviceProtection  7043 non-null   int64  
 11  TechSupport       7043 non-null   int64  
 12  StreamingTV       7043 non-null   int64  
 13  StreamingMovies   7043 non-null   int64  
 14  Contract          7043 non-null   int64  
 15  PaperlessBilling  7043 non-null   int64  
 16  PaymentMethod     7043 non-null   int64  


In [29]:
smote = SMOTE()
X_res , y_res = smote.fit_resample(X,y)


In [30]:
X_res

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,1,0,0,1,1,2,1,1,1,1,2,1,2,29.850000,29.850000
1,1,0,0,0,34,1,1,1,2,1,2,1,1,1,0,0,3,56.950000,1889.500000
2,1,0,0,0,2,1,1,1,2,2,1,1,1,1,2,1,3,53.850000,108.150000
3,1,0,0,0,45,0,0,1,2,1,2,2,1,1,0,0,0,42.300000,1840.750000
4,0,0,0,0,2,1,1,0,1,1,1,1,1,1,2,1,2,70.700000,151.650000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10343,1,0,0,0,1,1,1,1,1,1,2,1,1,1,2,0,3,50.646454,50.646454
10344,0,0,1,1,49,1,2,0,1,1,2,1,2,2,1,0,0,107.916191,5294.021569
10345,0,0,0,1,14,1,1,0,1,2,1,1,1,2,2,1,0,87.640802,1265.876907
10346,0,0,0,0,49,1,1,0,1,1,1,1,1,1,1,0,1,72.490868,3552.560370


In [31]:
X_train , X_test , y_train , y_test = train_test_split(X_res ,y_res , test_size = 0.25 , random_state = 42)

In [32]:
model_smote.fit(X_train , y_train)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3


In [33]:
y_pred_smote = model_smote.predict(X_test)
print(classification_report(y_test , y_pred_smote ))

              precision    recall  f1-score   support

           0       0.81      0.74      0.77      1298
           1       0.76      0.82      0.79      1289

    accuracy                           0.78      2587
   macro avg       0.78      0.78      0.78      2587
weighted avg       0.78      0.78      0.78      2587



# RandomForestClassifier


In [34]:
num_pipe = Pipeline( 
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'median')) ,
        ( 'scaler' , StandardScaler() )   
]) 

cat_pipe = Pipeline( 
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'most_frequent')) ,
        ( 'encoder' , ce.TargetEncoder() )   
]) 

preprocessor = ColumnTransformer(
    transformers = [
        ('num' , num_pipe , num ), 
        ('cat' , cat_pipe , cat )
])

model_smote_rf = Pipeline(
    steps = [
        ("preprocessor" , preprocessor) , 
        ('classifier' , RandomForestClassifier(n_estimators=50,criterion='gini') )
    ]
)

model_smote_rf.fit(X_train , y_train)
y_pred_smote_rf = model_smote_rf.predict(X_test)
print(classification_report(y_test , y_pred_smote_rf ))

              precision    recall  f1-score   support

           0       0.84      0.85      0.85      1298
           1       0.85      0.84      0.84      1289

    accuracy                           0.84      2587
   macro avg       0.84      0.84      0.84      2587
weighted avg       0.84      0.84      0.84      2587



In [35]:
# Paramters Grid Search Dictionary
param_grid = { 
    'classifier__n_estimators': [50,100,200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5, 10], 
    'classifier__max_features' : ["sqrt", "log2", None]
}

grid = GridSearchCV(model_smote_rf, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_


Fitting 5 folds for each of 81 candidates, totalling 405 fits


In [36]:
print(best_model)  # text repr: the HTML repr fails with category_encoders feature names

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   TargetEncoder())]),
                                    

In [37]:
""" 
param_grid = [
    {
        "Model1" : [RandomForestClassifier()] ,
        "model1__max_depth" : [10, 20, None] ,
        "model1__max_features" : ["sqrt", "log2", None]
    } , 

    {
        {
        "Model2" : [DecisionTreeClassifier()] ,
        "model2__max_depth" : [10, 20, None] ,
        "model2__max_features" : ["sqrt", "log2", None]
    } , 
    }
]

grid = GridSearchCV(model_smote, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
"""

' \nparam_grid = [\n    {\n        "Model1" : [RandomForestClassifier()] ,\n        "model1__max_depth" : [10, 20, None] ,\n        "model1__max_features" : ["sqrt", "log2", None]\n    } , \n\n    {\n        {\n        "Model2" : [DecisionTreeClassifier()] ,\n        "model2__max_depth" : [10, 20, None] ,\n        "model2__max_features" : ["sqrt", "log2", None]\n    } , \n    }\n]\n\ngrid = GridSearchCV(model_smote, param_grid, cv=5, scoring=\'f1\', n_jobs=-1, verbose=1)\ngrid.fit(X_train, y_train)\n\nbest_model = grid.best_estimator_\n'

In [38]:
y_pred_best_model = best_model.predict(X_test)
print(classification_report(y_test , y_pred_best_model ))

              precision    recall  f1-score   support

           0       0.84      0.85      0.84      1298
           1       0.85      0.84      0.84      1289

    accuracy                           0.84      2587
   macro avg       0.84      0.84      0.84      2587
weighted avg       0.84      0.84      0.84      2587



In [39]:
from sklearn.svm import LinearSVC

In [40]:
num_pipe = Pipeline( 
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'mean')) ,
        ( 'scaler' , StandardScaler() )   
]) 

cat_pipe = Pipeline( 
    steps = [
        ( 'imputer' , SimpleImputer(strategy = 'most_frequent')) ,
        ( 'encoder' , OneHotEncoder(handle_unknown='ignore') )   
]) 

preprocessor = ColumnTransformer(
    transformers = [
        ('num' , num_pipe , num ), 
        ('cat' , cat_pipe , cat )
])

model_smote_svc = Pipeline(
    steps = [
        ("preprocessor" , preprocessor) , 
        ('smote' , SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5) ) ,
        ('classifier' , LinearSVC() )
    ]
)

model_smote_svc.fit(X_train , y_train)
y_pred_smote_svc = model_smote_svc.predict(X_test)
print(classification_report(y_test , y_pred_smote_svc ))

              precision    recall  f1-score   support

           0       0.82      0.84      0.83      1298
           1       0.83      0.82      0.82      1289

    accuracy                           0.83      2587
   macro avg       0.83      0.83      0.83      2587
weighted avg       0.83      0.83      0.83      2587



# Search Model Tuning ( param Grid & Grid Search ) 

In [41]:
import joblib, json
from pathlib import Path

Path('models').mkdir(exist_ok=True)
joblib.dump(best_model, 'models/churn_pipeline.joblib')

metadata = {
    'feature_columns': X.columns.tolist(),
    'numerical_columns': num.tolist(),
    'categorical_columns': cat.tolist(),
    'label_mappings': label_mappings,   # categorical value -> integer code = its index in the list
    'target_classes': le_target.classes_.tolist(),   # index 0 / 1
    'best_params': grid.best_params_,
}
Path('models/metadata.json').write_text(json.dumps(metadata, indent=2, default=str))

2310